In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

In [ ]:
# ============================================================
# RUNTIME BENCHMARK LOGGER
# ============================================================

from time import perf_counter
from datetime import datetime
from pathlib import Path
import pandas as pd
import os

RUNTIME_FILE = Path("FINAL_experimental_runtime_summary.csv")

if RUNTIME_FILE.exists():
    RUNTIME_LOG = pd.read_csv(RUNTIME_FILE).to_dict("records")
else:
    RUNTIME_LOG = []


def start_timer(experiment_name):
    print("\n" + "=" * 75)
    print(f"STARTING: {experiment_name}")
    print("Clock time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 75)

    return perf_counter()


def stop_timer(experiment_name, start_time):
    elapsed_sec = perf_counter() - start_time

    record = {
        "experiment": experiment_name,
        "runtime_seconds": elapsed_sec,
        "runtime_minutes": elapsed_sec / 60.0,
        "runtime_hours": elapsed_sec / 3600.0,
    }

    RUNTIME_LOG.append(record)

    pd.DataFrame(RUNTIME_LOG).to_csv(
        RUNTIME_FILE,
        index=False
    )

    print("\n" + "-" * 75)
    print(f"FINISHED: {experiment_name}")
    print(f"Runtime: {elapsed_sec:.2f} seconds")
    print(f"Runtime: {elapsed_sec / 60.0:.4f} minutes")
    print("-" * 75)

    return elapsed_sec

In [ ]:
df = pd.read_csv("../Data/Raw/triage_cleaned_working.csv")
diag_train = pd.read_csv("../Data/Raw/triage_train_diagnostics.csv")

print(df.shape, diag_train.shape)

Helper function for slice metrics

In [ ]:
def slice_metrics(data, mask, y_col="y", p_col="oof_pred_proba", threshold=0.5):
    sub = data.loc[mask].copy()
    n = len(sub)
    
    if n == 0:
        return None
    
    y = sub[y_col].values
    p = sub[p_col].values
    y_pred = (p >= threshold).astype(int)
    
    # Safe metrics
    auroc = np.nan
    if len(np.unique(y)) > 1:
        auroc = roc_auc_score(y, p)
    
    auprc = average_precision_score(y, p)
    
    tn = fp = fn = tp = 0
    if len(np.unique(y)) > 1:
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    else:
        # handle single-class slices
        if y[0] == 1:
            tp = (y_pred == 1).sum()
            fn = (y_pred == 0).sum()
        else:
            tn = (y_pred == 0).sum()
            fp = (y_pred == 1).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    
    return {
        "n": n,
        "support_pct": 100 * n / len(data),
        "event_rate": y.mean(),
        "AUROC": auroc,
        "AUPRC": auprc,
        "Recall@0.5": recall,
        "Precision@0.5": precision,
        "avg_logloss": sub["oof_logloss"].mean(),
        "high_loss_rate": sub["high_loss_top10"].mean(),
        "high_conf_contradiction_rate": sub["high_conf_contradiction"].mean(),
        "fn_rate": sub["false_negative_05"].mean()
    }

Global reference metrics

In [ ]:
global_metrics = slice_metrics(diag_train, diag_train.index == diag_train.index)

global_metrics

In [ ]:
t_runtime_clinical = start_timer(
    "BASiC clinical candidate discovery"
)
diag_train["tachycardia_flag"] = (
    diag_train["heart_rate_final"].notna() & (diag_train["heart_rate_final"] > 100)
).astype(int)

diag_train["hypotension_flag"] = (
    diag_train["bp_systolic"].notna() & (diag_train["bp_systolic"] < 90)
).astype(int)

diag_train["tachypnea_flag"] = (
    diag_train["respiratory"].notna() & (diag_train["respiratory"] > 20)
).astype(int)

diag_train["hypoxia_flag"] = (
    diag_train["spo2"].notna() & (diag_train["spo2"] < 95)
).astype(int)

diag_train["fever_temp_flag"] = (
    diag_train["temperature"].notna() & (diag_train["temperature"] >= 38.0)
).astype(int)

In [ ]:
print(diag_train[
    ["tachycardia_flag", "hypotension_flag", "tachypnea_flag", "hypoxia_flag", "fever_temp_flag"]
].sum())

simple rule dictionary (single-variable slices)

In [ ]:
slice_defs = {
    # Complaint slices
    "chest_pain": lambda d: d["chest_pain"] == 1,
    "abdominal_pain": lambda d: d["abdominal_pain"] == 1,
    "back_pain": lambda d: d["back_pain"] == 1,
    "headache": lambda d: d["headache"] == 1,
    "cough": lambda d: d["cough"] == 1,
    "unclassified": lambda d: d["unclassified"] == 1,
    
    # Comorbidity slices
    "ckd": lambda d: d["ckd"] == 1,
    "dm": lambda d: d["dm"] == 1,
    "htn": lambda d: d["htn"] == 1,
    "chd": lambda d: d["chd"] == 1,
    "pneumonia": lambda d: d["pneumonia"] == 1,
    
    # Operational slices
    "arrival_mode_ambulance": lambda d: d["arrival_mode"] == "ambulance",
    "arrival_mode_wheelchair": lambda d: d["arrival_mode"] == "Wheelchair",
    "arrival_mode_unknown": lambda d: d["arrival_mode"] == "Unknown",
    "vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
    "spo2_missing": lambda d: d["spo2_missing"] == 1,
    "hr_missing": lambda d: d["hr_missing"] == 1,
    
    # Physiology slices
    "tachycardia": lambda d: d["tachycardia_flag"] == 1,
    "hypotension": lambda d: d["hypotension_flag"] == 1,
    "tachypnea": lambda d: d["tachypnea_flag"] == 1,
    "hypoxia": lambda d: d["hypoxia_flag"] == 1,
    "fever_temp": lambda d: d["fever_temp_flag"] == 1,
}

a few degree-2 slices

In [ ]:
slice_defs.update({
    "chest_pain_and_ambulance": lambda d: (d["chest_pain"] == 1) & (d["arrival_mode"] == "ambulance"),
    "abdominal_pain_and_hr_missing": lambda d: (d["abdominal_pain"] == 1) & (d["hr_missing"] == 1),
    "ckd_and_spo2_missing": lambda d: (d["ckd"] == 1) & (d["spo2_missing"] == 1),
    "pneumonia_and_tachypnea": lambda d: (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1),
    "unclassified_and_vitals_missing_ge_3": lambda d: (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 3),
})

Compute slice table

In [ ]:
rows = []

for slice_name, fn in slice_defs.items():
    mask = fn(diag_train)
    m = slice_metrics(diag_train, mask)
    
    if m is not None:
        m["slice_name"] = slice_name
        m["auprc_gap_vs_global"] = m["AUPRC"] - global_metrics["AUPRC"]
        m["recall_gap_vs_global"] = m["Recall@0.5"] - global_metrics["Recall@0.5"]
        m["logloss_lift_vs_global"] = m["avg_logloss"] - global_metrics["avg_logloss"]
        rows.append(m)

slice_table = pd.DataFrame(rows)

slice_table = slice_table[
    [
        "slice_name", "n", "support_pct", "event_rate",
        "AUROC", "AUPRC", "Recall@0.5", "Precision@0.5",
        "avg_logloss", "high_loss_rate", "high_conf_contradiction_rate", "fn_rate",
        "auprc_gap_vs_global", "recall_gap_vs_global", "logloss_lift_vs_global"
    ]
]

slice_table.sort_values("logloss_lift_vs_global", ascending=False).head(20)

first-pass support filter

In [ ]:
min_support_pct = 0.5

slice_table_supported = slice_table[slice_table["support_pct"] >= min_support_pct].copy()

print("Supported slices:", len(slice_table_supported))
slice_table_supported.sort_values("logloss_lift_vs_global", ascending=False)

Rank slices by “harm”

In [ ]:
slice_table_supported["priority_score"] = (
    2.0 * slice_table_supported["logloss_lift_vs_global"].fillna(0) +
    1.5 * (-slice_table_supported["recall_gap_vs_global"].fillna(0)) +
    1.0 * slice_table_supported["high_conf_contradiction_rate"].fillna(0) +
    1.0 * slice_table_supported["high_loss_rate"].fillna(0)
)

slice_table_ranked = slice_table_supported.sort_values("priority_score", ascending=False)

slice_table_ranked.head(20)

Tag slice types

In [ ]:
def tag_slice_type(name):
    if any(x in name for x in ["arrival_mode", "missing", "vitals_missing", "unknown"]):
        return "operational_or_missingness"
    elif any(x in name for x in ["tachycardia", "hypotension", "tachypnea", "hypoxia", "fever"]):
        return "physiologic"
    elif any(x in name for x in ["ckd", "dm", "htn", "chd", "pneumonia"]):
        return "comorbidity"
    else:
        return "complaint_or_mixed"

slice_table_ranked["slice_type"] = slice_table_ranked["slice_name"].apply(tag_slice_type)
slice_table_ranked.head(20)

Save candidate clinical slice table

In [ ]:
slice_table_ranked.to_csv("clinical_slice_candidates.csv", index=False)
print("Saved clinical_slice_candidates.csv")
stop_timer(
    "BASiC clinical candidate discovery",
    t_runtime_clinical
)

Select top slices for the next notebook

In [ ]:
top_slices = slice_table_ranked.head(10)[["slice_name", "slice_type", "n", "support_pct", "priority_score"]]
top_slices